# Emotion Detection Project (FER2013)Pipeline includes:1. Data Loading2. Data Augmentation3. Improved CNN Model4. Training with Callbacks5. Evaluation + Confusion Matrix6. Transfer Learning (MobileNetV2)7. Model Comparison8. Webcam Emotion Detection DemoDataset structure expected:```dataset/   train/      angry/      happy/      sad/      ...   test/      angry/      happy/      sad/```

In [ ]:
import tensorflow as tffrom tensorflow.keras import layers, modelsimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import confusion_matrix, classification_report

## Load Dataset

In [ ]:
IMG_SIZE = (48,48)BATCH_SIZE = 64train_data = tf.keras.preprocessing.image_dataset_from_directory(    "dataset/train",    image_size=IMG_SIZE,    color_mode="grayscale",    batch_size=BATCH_SIZE)test_data = tf.keras.preprocessing.image_dataset_from_directory(    "dataset/test",    image_size=IMG_SIZE,    color_mode="grayscale",    batch_size=BATCH_SIZE,    shuffle=False)class_names = train_data.class_namesprint(class_names)

## Data Augmentation

In [ ]:
data_aug = tf.keras.Sequential([    layers.RandomFlip("horizontal"),    layers.RandomRotation(0.1),    layers.RandomZoom(0.1),])

## Improved CNN Model

In [ ]:
cnn_model = models.Sequential([    layers.Input(shape=(48,48,1)),    data_aug,    layers.Conv2D(32,(3,3),padding="same",activation="relu"),    layers.BatchNormalization(),    layers.Conv2D(32,(3,3),padding="same",activation="relu"),    layers.BatchNormalization(),    layers.MaxPooling2D(),    layers.Dropout(0.25),    layers.Conv2D(64,(3,3),padding="same",activation="relu"),    layers.BatchNormalization(),    layers.Conv2D(64,(3,3),padding="same",activation="relu"),    layers.BatchNormalization(),    layers.MaxPooling2D(),    layers.Dropout(0.25),    layers.Conv2D(128,(3,3),padding="same",activation="relu"),    layers.BatchNormalization(),    layers.MaxPooling2D(),    layers.Dropout(0.3),    layers.Flatten(),    layers.Dense(256,activation="relu"),    layers.BatchNormalization(),    layers.Dropout(0.5),    layers.Dense(len(class_names),activation="softmax")])cnn_model.compile(    optimizer="adam",    loss="sparse_categorical_crossentropy",    metrics=["accuracy"])cnn_model.summary()

## Training CNN

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(    monitor="val_loss",    patience=5,    restore_best_weights=True)reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(    monitor="val_loss",    factor=0.3,    patience=3,    min_lr=1e-5)history = cnn_model.fit(    train_data,    validation_data=test_data,    epochs=40,    callbacks=[early_stop, reduce_lr])

## Save CNN Model

In [ ]:
cnn_model.save("emotion_cnn_model.h5")

## Confusion Matrix

In [ ]:
y_true = []y_pred = []for images, labels in test_data:    preds = cnn_model.predict(images)    preds = np.argmax(preds, axis=1)        y_true.extend(labels.numpy())    y_pred.extend(preds)cm = confusion_matrix(y_true, y_pred)plt.figure(figsize=(8,6))sns.heatmap(cm, annot=True, fmt="d")plt.xlabel("Predicted")plt.ylabel("Actual")plt.title("CNN Confusion Matrix")plt.show()print(classification_report(y_true,y_pred,target_names=class_names))

## Transfer Learning (MobileNetV2)

In [ ]:
IMG_SIZE_TL = (96,96)train_tl = tf.keras.preprocessing.image_dataset_from_directory(    "dataset/train",    image_size=IMG_SIZE_TL,    batch_size=BATCH_SIZE)test_tl = tf.keras.preprocessing.image_dataset_from_directory(    "dataset/test",    image_size=IMG_SIZE_TL,    batch_size=BATCH_SIZE)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(    input_shape=(96,96,3),    include_top=False,    weights="imagenet")base_model.trainable = Falsetl_model = models.Sequential([    base_model,    layers.GlobalAveragePooling2D(),    layers.BatchNormalization(),    layers.Dense(128,activation="relu"),    layers.Dropout(0.5),    layers.Dense(len(class_names),activation="softmax")])tl_model.compile(    optimizer="adam",    loss="sparse_categorical_crossentropy",    metrics=["accuracy"])tl_model.summary()

In [ ]:
history_tl = tl_model.fit(    train_tl,    validation_data=test_tl,    epochs=20)

## Save Transfer Learning Model

In [ ]:
tl_model.save("emotion_transfer_model.h5")

## Webcam Emotion Detection (OpenCV)

In [ ]:
import cv2model = tf.keras.models.load_model("emotion_cnn_model.h5")face_cascade = cv2.CascadeClassifier(    cv2.data.haarcascades + "haarcascade_frontalface_default.xml")cap = cv2.VideoCapture(0)while True:    ret, frame = cap.read()        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)        faces = face_cascade.detectMultiScale(gray,1.3,5)        for (x,y,w,h) in faces:        face = gray[y:y+h,x:x+w]        face = cv2.resize(face,(48,48))        face = face/255.0        face = np.reshape(face,(1,48,48,1))                pred = model.predict(face)        emotion = class_names[np.argmax(pred)]                cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)        cv2.putText(frame,emotion,(x,y-10),                    cv2.FONT_HERSHEY_SIMPLEX,0.9,(0,255,0),2)            cv2.imshow("Emotion Detector",frame)        if cv2.waitKey(1) & 0xFF == ord('q'):        breakcap.release()cv2.destroyAllWindows()